In [ ]:
import pandas as pd
import numpy as np

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Paths
raw_path = '../data/raw/online_retail_II.csv'
processed_path = '../data/processed/cleaned_online_retail.csv'

# Load raw dataset
df = pd.read_csv(raw_path, low_memory=False)
initial_rows = len(df)
print(f"Loaded raw dataset with {initial_rows:,} rows.")

In [ ]:
# Strip whitespace from column names and replace spaces with underscores
df.columns = df.columns.str.strip().str.replace(' ', '_')

# Parse InvoiceDate to standard datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Clean string columns: strip whitespace from StockCode, Description, and Country
for col in ['StockCode', 'Description', 'Country']:
    df[col] = df[col].astype(str).str.strip()

print("Column names and date formats standardized.")
df.head(2)

In [ ]:
# Audit and drop exact duplicate records
duplicate_count = df.duplicated().sum()
print(f"Exact duplicate rows detected: {duplicate_count:,} ({(duplicate_count/initial_rows)*100:.2f}%)")

# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows remaining after duplicate removal: {len(df):,}")

In [ ]:
missing_customers = df['Customer_ID'].isnull().sum()
print(f"Rows missing Customer_ID: {missing_customers:,} ({(missing_customers/len(df))*100:.2f}%)")

# Fill missing Customer_ID with a placeholder (-1) instead of dropping rows
df['Customer_ID'] = df['Customer_ID'].fillna(-1).astype(int)
print("Missing Customer_ID records retained and labeled as -1 (Guest Checkouts).")

In [ ]:
# Identify non-inventory codes (administrative fees, manual adjustments, postage)
non_product_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT', 'CRUK']

# Find rows matching administrative codes or short non-standard codes
invalid_code_mask = (
    df['StockCode'].str.upper().isin(non_product_codes) |
    (df['StockCode'].str.len() < 4)
)

invalid_code_count = invalid_code_mask.sum()
print(f"Non-product / administrative records identified: {invalid_code_count:,}")

# Filter out non-product transactions
df = df[~invalid_code_mask].reset_index(drop=True)
print(f"Rows remaining after filtering non-product codes: {len(df):,}")

In [ ]:
# Check for zero or negative prices
zero_neg_price_mask = df['Price'] <= 0
invalid_price_count = zero_neg_price_mask.sum()

print(f"Records with Price <= 0: {invalid_price_count:,}")

# Inspect sample descriptions of zero-price items (often sample giveaways or system errors)
print("\nSample descriptions for zero/negative price items:")
print(df[zero_neg_price_mask]['Description'].value_counts().head(5))

# Filter out records with Price <= 0
df = df[df['Price'] > 0].reset_index(drop=True)
print(f"\nRows remaining after filtering invalid prices: {len(df):,}")

In [ ]:
# Create explicit flags for cancelled orders
df['Is_Cancelled'] = df['Invoice'].astype(str).str.startswith('C') | (df['Quantity'] < 0)

cancelled_count = df['Is_Cancelled'].sum()
print(f"Total cancelled/return transaction lines: {cancelled_count:,} ({(cancelled_count/len(df))*100:.2f}%)")

# Separate regular sales vs returned units for clear business logging
df['Quantity_Sold'] = np.where(df['Quantity'] > 0, df['Quantity'], 0)
df['Quantity_Returned'] = np.where(df['Quantity'] < 0, np.abs(df['Quantity']), 0)

# Net Quantity directly impacts warehouse depletion
df['Net_Quantity'] = df['Quantity']

# Calculate total revenue line item (Net_Quantity * Price)
df['Line_Revenue'] = df['Net_Quantity'] * df['Price']

print("Created net demand metrics: Quantity_Sold, Quantity_Returned, Net_Quantity, and Line_Revenue.")

In [ ]:
# Final validation summary table
final_rows = len(df)
removed_rows = initial_rows - final_rows

summary_df = pd.DataFrame({
    'Metric': [
        'Initial Raw Rows',
        'Final Cleaned Rows',
        'Total Rows Removed',
        'Percentage Retained',
        'Unique Products (StockCodes)',
        'Unique Invoices',
        'Total Net Revenue (£)',
        'Total Net Units Sold'
    ],
    'Value': [
        f"{initial_rows:,}",
        f"{final_rows:,}",
        f"{removed_rows:,}",
        f"{(final_rows / initial_rows) * 100:.2f}%",
        f"{df['StockCode'].nunique():,}",
        f"{df['Invoice'].nunique():,}",
        f"£{df['Line_Revenue'].sum():,.2f}",
        f"{df['Net_Quantity'].sum():,}"
    ]
})

print("=== DATA CLEANING PIPELINE SUMMARY ===")
display(summary_df)

In [ ]:
# Export cleaned dataset for downstream SQL and Time Series modules
df.to_csv(processed_path, index=False)
print(f"Cleaned dataset successfully saved to: {processed_path}")